# HighRes LidValet

The HighRes Biosolutions **LidValet** is a benchtop delidder. Each of its nests (the
controller calls them *hotels*) holds a vacuum suction cup above a plate position: a mover
presents a lidded plate to a nest, the LidValet lifts the lid off into the cup and holds it
there, and later lowers it back onto the plate.

| | |
|---|---|
| Communication | TCP socket |
| Default address | `192.168.127.60:1000` (printed on the back of the device) |
| Protocol | newline-terminated ASCII, `ACK!` / `OK!` / `ERROR!` |
| Nests | read from the device (`ACTIVE_HOTELS`) |
| Nest states | `open`, `has_lid`, `busy`, `error`, `unknown` |

[OEM link](https://www.highres.com/lab-instruments/lid-management)

## Physical setup

The LidValet is a network device: connect it to the host with an Ethernet cable, either
directly or through a switch. It does **not** run a DHCP client, so give the host an address
on the device's subnet, for example:

```
sudo ip addr add 192.168.127.1/24 dev <interface>
```

The address on the label is the one to talk to. A unit that has been sitting powered but
idle can hold link while answering nothing at all — if it is silent, power-cycle it and it
will announce itself on boot.

## Connect

`setup()` opens the socket, reads the nest count off the device, refuses to start if a lid
is stuck in a suction cup, and resets every nest. Pass `host` and `port` for your machine —
the nest count is not something you configure, it comes from the device itself.

In [ ]:
from pylabrobot.high_res import HighResLidValet

valet = HighResLidValet(host="192.168.127.60", port=1000)
await valet.setup()

print(valet.num_nests)

## Nest state

`request_state()` returns one nest's state; `request_all_states()` returns every nest in a
single round trip. `request_has_lid()` and `request_is_busy()` are convenience wrappers.

A nest reports `error` when its last operation failed — it stays that way until the nest is
reset.

In [ ]:
print(await valet.request_state(1))
print(await valet.request_all_states())

## Delid a plate

With a lidded plate on a nest, `delid()` lifts the lid into that nest's suction cup. The cup
must be empty; the call raises if it is already holding a lid or if the nest is busy.

In [ ]:
await valet.delid(nest=1)

## Re-lid the plate

`lid()` lowers the held lid back onto the plate. The cup must be holding a lid. It retries a
few times to ride out a transient busy state, matching the device server's own behaviour.

In [ ]:
await valet.lid(nest=1)

## Reset

`reset()` homes a nest and clears its error state. Called with no argument it resets every
nest in one command, which is also what `setup()` does.

In [ ]:
await valet.reset(nest=1)
await valet.reset()

## Homing and lift motion

`home()` homes the whole system. `wave()` drops and raises the lifts a given number of times
— useful for confirming the machine is alive and that the pneumatics are connected. Both
block until the motion is finished.

In [ ]:
await valet.home()
await valet.wave(cycles=3)

## Vacuum and purge

The suction cups are driven by solenoid outputs, which `delid()` and `lid()` operate for
you. `set_vacuum()` and `set_purge()` drive them directly, which is mainly useful when
diagnosing a cup that will not pick up.

These return immediately: the controller switches the output and does not wait, and on a
machine with no vacuum sensor fitted it cannot confirm suction was achieved.

In [ ]:
await valet.set_vacuum(nest=1, on=True)
await valet.set_vacuum(nest=1, on=False)

## Diagnostics

The controller keeps an error stack and a command history, and reports its own versions.
When something fails, `request_errors()` is the first place to look — the device's own message
is more specific than the exception text.

In [ ]:
print(await valet.request_version())
print(await valet.request_errors(5))

## Settings

The device exposes its full calibration and configuration. `request_settings()` returns it as a
typed, frozen object, one attribute per device setting.

In [ ]:
settings = await valet.request_settings()
print(settings.serial_number, settings.active_hotels)

## Disconnect

`stop()` asks the server to drop this client and then closes the socket, so the controller
frees the connection slot rather than waiting for it to time out.

In [ ]:
await valet.stop()